# Chapter 12 - Medium Tasks

In these tasks, you'll experiment with different QLoRA configurations and compare their effects on model performance and efficiency.

---

### Setup

In [ ]:
# Install packages (uncomment if on Colab)
# %%capture
# !pip install -q accelerate==0.31.0 peft==0.11.1 bitsandbytes==0.43.1 transformers==4.41.2 trl==0.9.4 sentencepiece==0.2.0

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, AutoPeftModelForCausalLM
from trl import SFTTrainer
from datasets import load_dataset
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

### Load and Prepare Data

In [ ]:
# Load tokenizer and format data
template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")
      .shuffle(seed=42)
      .select(range(1_500))
)
dataset = dataset.map(format_prompt)
print(f"Dataset loaded: {len(dataset)} examples")

---

## Task 1: Compare Different LoRA Ranks

The rank (r) parameter in LoRA controls the dimensionality of the adapter matrices. Higher rank means more parameters.

**Your task:** Train models with different ranks and compare:
- Number of trainable parameters
- Training time
- Memory usage
- Generation quality

**Instructions:**
1. Train three models with ranks: r=8, r=32, r=64
2. Record metrics for each
3. Test each model with the same prompts
4. Compare the results

In [ ]:
def train_with_rank(rank, max_steps=50, output_name="model"):
    """Train a model with specified LoRA rank and return metrics."""
    
    model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
    
    # Quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="float16",
        bnb_4bit_use_double_quant=True,
    )
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=bnb_config,
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = "<PAD>"
    tokenizer.padding_side = "left"
    
    # TODO: Configure LoRA with the specified rank
    peft_config = LoraConfig(
        lora_alpha=# YOUR CODE HERE,  # Set to 2*rank
        lora_dropout=0.1,
        r=# YOUR CODE HERE,  # Use the rank parameter
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
    )
    
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, peft_config)
    
    # Get trainable parameter count
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    
    # Training args
    training_args = TrainingArguments(
        output_dir=f"./results_{output_name}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        optim="paged_adamw_32bit",
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        max_steps=max_steps,
        logging_steps=10,
        fp16=True,
        gradient_checkpointing=True
    )
    
    # Train
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        dataset_text_field="text",
        tokenizer=tokenizer,
        args=training_args,
        max_seq_length=512,
        peft_config=peft_config,
    )
    
    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time
    
    # Save
    trainer.model.save_pretrained(output_name)
    
    return {
        "rank": rank,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_percent": 100 * trainable_params / total_params,
        "training_time": training_time,
        "model_path": output_name
    }

In [ ]:
# TODO: Train models with different ranks
results = []

# Rank 8
print("Training with rank=8...")
metrics_r8 = train_with_rank(rank=# YOUR CODE HERE, output_name="tinyllama-r8")
results.append(metrics_r8)

# Rank 32
print("\nTraining with rank=32...")
metrics_r32 = train_with_rank(rank=# YOUR CODE HERE, output_name="tinyllama-r32")
results.append(metrics_r32)

# Rank 64
print("\nTraining with rank=64...")
metrics_r64 = train_with_rank(rank=# YOUR CODE HERE, output_name="tinyllama-r64")
results.append(metrics_r64)

In [ ]:
# Compare results
df_results = pd.DataFrame(results)
print("\nComparison of Different Ranks:")
print("="*80)
print(df_results.to_string(index=False))

In [ ]:
# TODO: Test all three models with the same prompt
test_prompt = """<|user|>
Explain the difference between supervised and unsupervised learning.</s>
<|assistant|>
"""

for result in results:
    print(f"\n{'='*80}")
    print(f"Model with rank={result['rank']}")
    print(f"{'='*80}")
    
    # Load and test
    model = AutoPeftModelForCausalLM.from_pretrained(
        result['model_path'],
        low_cpu_mem_usage=True,
        device_map="auto",
    )
    merged_model = model.merge_and_unload()
    
    tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T")
    pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer, max_new_tokens=100)
    
    output = pipe(test_prompt)[0]['generated_text']
    print(output[len(test_prompt):])
    
    # Clean up memory
    del model, merged_model, pipe
    torch.cuda.empty_cache()

### Questions:

**Q1:** How does increasing the rank affect the number of trainable parameters?

*Your answer here*

**Q2:** Is there a noticeable difference in generation quality between different ranks? Which rank seems best?

*Your answer here*

**Q3:** What trade-offs should you consider when choosing a rank?

*Your answer here*

---

## Task 2: Experiment with Target Modules

LoRA can be applied to different modules in the transformer. Common targets include:
- Query/Key/Value projections (q_proj, k_proj, v_proj)
- Output projection (o_proj)
- Feed-forward layers (gate_proj, up_proj, down_proj)

**Your task:** Compare applying LoRA to different module combinations.

**Instructions:**
1. Train a model with LoRA only on attention (q_proj, k_proj, v_proj, o_proj)
2. Train a model with LoRA only on feed-forward (gate_proj, up_proj, down_proj)
3. Train a model with LoRA on all modules
4. Compare the results

In [ ]:
def train_with_target_modules(target_modules, max_steps=50, output_name="model"):
    """Train a model with specified target modules."""
    
    model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="float16",
        bnb_4bit_use_double_quant=True,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=bnb_config,
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = "<PAD>"
    tokenizer.padding_side = "left"
    
    # TODO: Configure LoRA with the specified target modules
    peft_config = LoraConfig(
        lora_alpha=32,
        lora_dropout=0.1,
        r=16,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=# YOUR CODE HERE  # Use the target_modules parameter
    )
    
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, peft_config)
    
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    training_args = TrainingArguments(
        output_dir=f"./results_{output_name}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        optim="paged_adamw_32bit",
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        max_steps=max_steps,
        logging_steps=10,
        fp16=True,
        gradient_checkpointing=True
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        dataset_text_field="text",
        tokenizer=tokenizer,
        args=training_args,
        max_seq_length=512,
        peft_config=peft_config,
    )
    
    trainer.train()
    trainer.model.save_pretrained(output_name)
    
    return {
        "modules": ", ".join(target_modules),
        "num_modules": len(target_modules),
        "trainable_params": trainable_params,
        "model_path": output_name
    }

In [ ]:
# TODO: Train with different target module combinations
module_results = []

# Attention only
print("Training with attention modules only...")
attention_modules = # YOUR CODE HERE  # ['q_proj', 'k_proj', 'v_proj', 'o_proj']
metrics_attn = train_with_target_modules(attention_modules, output_name="tinyllama-attn")
module_results.append(metrics_attn)

# Feed-forward only
print("\nTraining with feed-forward modules only...")
ffn_modules = # YOUR CODE HERE  # ['gate_proj', 'up_proj', 'down_proj']
metrics_ffn = train_with_target_modules(ffn_modules, output_name="tinyllama-ffn")
module_results.append(metrics_ffn)

# All modules
print("\nTraining with all modules...")
all_modules = # YOUR CODE HERE  # Combine both lists
metrics_all = train_with_target_modules(all_modules, output_name="tinyllama-all")
module_results.append(metrics_all)

In [ ]:
# Compare results
df_modules = pd.DataFrame(module_results)
print("\nComparison of Different Target Modules:")
print("="*80)
print(df_modules.to_string(index=False))

### Questions:

**Q4:** Which module combination has the most trainable parameters? Why?

*Your answer here*

**Q5:** Based on the parameter counts, which approach offers the best efficiency?

*Your answer here*

---

## Task 3: Compare Quantization Settings

Quantization is a key component of QLoRA. Let's compare:
- 4-bit quantization
- 8-bit quantization  
- No quantization (if memory allows)

**Your task:** Compare different quantization settings.

**Instructions:**
1. Train with 4-bit quantization (NF4)
2. Train with 8-bit quantization
3. Compare memory usage and training speed

**Note:** Be careful with memory! You may need to reduce batch size or skip no-quantization depending on your GPU.

In [ ]:
# TODO: Create function to train with different quantization settings
def train_with_quantization(load_in_4bit=True, load_in_8bit=False, max_steps=30, output_name="model"):
    """Train with specified quantization settings."""
    
    model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
    
    # TODO: Configure quantization
    if load_in_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype="float16",
            bnb_4bit_use_double_quant=True,
        )
    elif load_in_8bit:
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=# YOUR CODE HERE,
        )
    else:
        bnb_config = None
    
    # YOUR CODE HERE: Load model with appropriate quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=bnb_config,
    )
    
    # Rest of training code...
    # (Complete this function following the previous examples)
    pass

### Questions:

**Q6:** What is the main benefit of using 4-bit quantization over 8-bit?

*Your answer here*

**Q7:** In what scenarios might you prefer 8-bit over 4-bit quantization?

*Your answer here*

---

## Bonus: Optimal Configuration

Based on your experiments, what would be your recommended LoRA configuration for:
1. **Maximum Quality** (ignoring efficiency)
2. **Maximum Efficiency** (smallest, fastest)
3. **Balanced** (best quality/efficiency trade-off)

Document your recommendations below with justification:

### Your Recommendations:

**Maximum Quality:**
- Rank: *your choice*
- Target modules: *your choice*
- Quantization: *your choice*
- Justification: *explain*

**Maximum Efficiency:**
- Rank: *your choice*
- Target modules: *your choice*
- Quantization: *your choice*
- Justification: *explain*

**Balanced:**
- Rank: *your choice*
- Target modules: *your choice*
- Quantization: *your choice*
- Justification: *explain*